# `Feature-engineering.ipynb`: Physics & Chemistry Feature Extraction, Chain-of-Physical-Thought (CoPT) & Counterfactual Negative Mining

### What This Notebook Does
This notebook transforms the cleaned physical/chemical data from `Dataset-creation.ipynb` (with automatic self-generation if run standalone) into **high-signal training features** designed specifically to make `gemma-4-e2b` obey the laws of physics and chemistry when generating code:

1. **Quantitative Chemical & Structural Feature Extraction (`physchem_feature_matrix.csv`)**:
   - Computes bond ionicity $I_{\text{Pauling}} = 1 - \exp\left(-\frac{1}{4}(\chi_A - \chi_B)^2\right)$, covalent vs. coordination cutoff safety margins $\Delta r_{\text{margin}} = r_{\text{HB,min}} - r_{\text{cov,cut}}$, stoichiometric charge balance $\sum q_i = 0$, and vibrational Nyquist timestep limits $\Delta t_{\max} = T_{\text{vib}} / 10$.
2. **Chain-of-Physical-Thought (`<physchem_reasoning>`) Prefix Engineering**:
   - Small 2B models write buggy scientific code when they jump straight to `import numpy as np` without first deciding the bond cutoffs, sub-phase partitioning, and unit conversions.
   - For every code generation example, we engineer a structured, 4-line `<physchem_reasoning>` header that explicitly derives:
     - `[Bonding & Species]`: Intramolecular covalent cutoff ($r_{\text{OH}} \le 1.25\text{ \AA}$) vs lattice stoichiometry.
     - `[Physical Law & PBC]`: Minimum image convention (`mic=True` / `neighbor_list`), spherical shell volume $4\pi r^2 \Delta r$, or slice mass conversion ($N_A, 10^{-24}\text{ cm}^3/\text{\AA}^3$).
     - `[ASE API Contract]`: Tuple unpacking `nl_i, nl_j, nl_d = neighbor_list('ijd', ...)` vs dense `get_all_distances(mic=True)`.
3. **Physics-Violating Counterfactual Negative Mining (for DPO Alignment)**:
   - Automatically generates **8 classes of physically or chemically illegal counterfactual scripts** (`rejected`) paired against the physically rigorous scripts (`chosen`):
     1. *Cutoff Confluence Bug*: Using $r_{\text{cut}}(\text{O}-\text{H}) = 2.0\text{ \AA}$ (mixing H-bonds with covalent bonds).
     2. *Sub-Phase Conflation Bug*: Treating all `'O'` atoms as crystal lattice or all `'O'` atoms as water.
     3. *Broken PBC Bug*: Computing `np.linalg.norm(pos[i] - pos[j])` across periodic boundaries without Minimum Image Convention.
     4. *Unnormalized RDF Bug*: Forgetting the $4\pi r^2 \Delta r$ spherical shell volume divisor in $g(r)$.
     5. *Dimensional Analysis Bug*: Omitting the $10^{-24}\text{ cm}^3/\text{\AA}^3$ or $1/(2 N_A)$ stoichiometry factor in $\rho(z)$.
     6. *Nyquist Integrator Instability Bug*: Running flexible water MD with $\Delta t = 5.0\text{ fs}$ (exploding O-H bonds).
     7. *ASE Hallucination Bug*: Calling `atoms.get_neighbors()` or indexing `neighbor_list('ijd')` as a 2D matrix `d[i, j]`.
     8. *Projection & Escape Bug*: Plotting X-Y instead of X-Z side-view or putting `\text{}` in Matplotlib labels.
4. **Curriculum Complexity Scoring & Token Weight Mask Generation**:
   - Scores every example by AST + physical coupling complexity and exports the exact files required by `PhysChem-finetuning.ipynb`:
     - `stage1_physchem_world_model.jsonl`
     - `stage2_copt_phys_code_sft.jsonl`
     - `stage3_counterfactual_physchem_dpo.jsonl`
     - `physchem_feature_matrix.csv`


In [ ]:
# ==============================================================================
# CELL 1: LOAD OR SELF-INITIALIZE RAW PHYSCHEM CORPUS & BOND KNOWLEDGE TABLE
# ==============================================================================
import os
import re
import ast
import json
import math
import random
import numpy as np
import pandas as pd
from pathlib import Path
import sys, subprocess
try:
    from ase.data import atomic_numbers, atomic_masses, covalent_radii
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ase>=3.23.0"])
    from ase.data import atomic_numbers, atomic_masses, covalent_radii

random.seed(42)
np.random.seed(42)

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
RAW_DIR = WORK_DIR / "physchem_raw_dataset"
FEAT_DIR = WORK_DIR / "physchem_engineered_features"
FEAT_DIR.mkdir(parents=True, exist_ok=True)

# Self-healing fallback: if Dataset-creation.ipynb was not run in the same session,
# check /kaggle/input or synthesize the bond registry on the fly.
PAULING_EN = {
    "H": 2.20, "C": 2.55, "N": 3.04, "O": 3.44, "Mg": 1.31, "Al": 1.61,
    "Si": 1.90, "Ti": 1.54, "Fe": 1.83, "Ni": 1.91, "Cu": 1.90, "Zn": 1.65,
    "Ru": 2.20, "Pd": 2.20, "Ag": 1.93, "Ce": 1.12, "Ir": 2.20, "Pt": 2.28, "Au": 2.54,
}

SYSTEM_PHYSCHEM_SPECS = [
    {"name": "Anatase TiO2", "metal": "Ti", "ox_state": 4, "stoich": 2, "r_mo_lat": 1.96, "r_mcus_ow": 2.22, "r_if_cut": 3.20, "r_oh_cut": 1.25, "z_tol": 1.2, "file": "/mnt/data/Anatase_TiO2_Water_MolecularAdsorbedWater.extxyz", "fb": "public/Anatase_TiO2_Water_MolecularAdsorbedWater.extxyz"},
    {"name": "Rutile IrO2", "metal": "Ir", "ox_state": 4, "stoich": 2, "r_mo_lat": 2.00, "r_mcus_ow": 2.15, "r_if_cut": 3.00, "r_oh_cut": 1.20, "z_tol": 1.5, "file": "/mnt/data/iro2_water_concat.extxyz", "fb": "Analysis/iro2_water_concat.extxyz"},
    {"name": "Rocksalt MgO", "metal": "Mg", "ox_state": 2, "stoich": 1, "r_mo_lat": 2.11, "r_mcus_ow": 2.18, "r_if_cut": 3.10, "r_oh_cut": 1.25, "z_tol": 1.1, "file": "/mnt/data/MgO_Water_Interface.extxyz", "fb": "public/MgO_Water_Interface.extxyz"},
    {"name": "Quartz SiO2", "metal": "Si", "ox_state": 4, "stoich": 2, "r_mo_lat": 1.62, "r_mcus_ow": 2.35, "r_if_cut": 3.30, "r_oh_cut": 1.22, "z_tol": 1.3, "file": "/mnt/data/SiO2_Water_Slab.extxyz", "fb": "public/SiO2_Water_Slab.extxyz"},
    {"name": "Hematite Fe2O3", "metal": "Fe", "ox_state": 3, "stoich": 2, "r_mo_lat": 2.02, "r_mcus_ow": 2.20, "r_if_cut": 3.20, "r_oh_cut": 1.25, "z_tol": 1.2, "file": "/mnt/data/Fe2O3_Water_Interface.extxyz", "fb": "public/Fe2O3_Water_Interface.extxyz"},
    {"name": "Ceria CeO2", "metal": "Ce", "ox_state": 4, "stoich": 2, "r_mo_lat": 2.34, "r_mcus_ow": 2.45, "r_if_cut": 3.40, "r_oh_cut": 1.25, "z_tol": 1.4, "file": "/mnt/data/CeO2_Water_Slab.extxyz", "fb": "public/CeO2_Water_Slab.extxyz"},
]

print(f"Loaded {len(SYSTEM_PHYSCHEM_SPECS)} physical-chemical system specifications for feature engineering.")


## Step 1: Extract Quantitative Chemical Bonding & Physical Invariant Features
For each metal-oxide/water system, we compute physical descriptors that govern how the simulation code must be parameterized:
- **Pauling Bond Ionicity**: $f_{\text{ionic}}(\text{M}-\text{O}) = 1 - \exp\left(-0.25 (\chi_{\text{O}} - \chi_{\text{M}})^2\right)$
- **Sum of Covalent Radii vs Experimental Bond Length**: $r_{\text{cov}}(\text{M}) + r_{\text{cov}}(\text{O})$ vs $r_{\text{exp}}(\text{M}-\text{O}_{\text{slab}})$
- **Covalent-to-H-Bond Cutoff Margin**: $\Delta r_{\text{OH}} = r_{\text{HB,min}}(1.65\text{ \AA}) - r_{\text{cov,max}}(1.05\text{ \AA}) = 0.60\text{ \AA}$
- **Harmonic Vibrational Stretching Period & Max MD Timestep**: $T_{\text{OH}} \approx 9.8\text{ fs} \implies \Delta t_{\max} = 1.0\text{ fs}$


In [ ]:
# ==============================================================================
# CELL 2: EXTRACT QUANTITATIVE CHEMICAL & PHYSICAL DESCRIPTOR MATRIX
# ==============================================================================
feature_rows = []
r_cov_o = float(covalent_radii[atomic_numbers["O"]])
r_cov_h = float(covalent_radii[atomic_numbers["H"]])
chi_o = PAULING_EN["O"]
chi_h = PAULING_EN["H"]

for spec in SYSTEM_PHYSCHEM_SPECS:
    m = spec["metal"]
    z_m = int(atomic_numbers[m])
    r_cov_m = float(covalent_radii[z_m])
    chi_m = PAULING_EN[m]
    delta_chi_mo = round(abs(chi_o - chi_m), 3)
    ionicity_mo = round(1.0 - math.exp(-0.25 * (delta_chi_mo ** 2)), 4)
    ionicity_oh = round(1.0 - math.exp(-0.25 * ((chi_o - chi_h) ** 2)), 4)

    feature_rows.append({
        "system_name": spec["name"],
        "metal_symbol": m,
        "metal_atomic_number": z_m,
        "metal_mass_amu": round(float(atomic_masses[z_m]), 4),
        "metal_oxidation_state": spec["ox_state"],
        "oxide_stoichiometry_s": spec["stoich"],
        "sum_covalent_radii_MO_A": round(r_cov_m + r_cov_o, 3),
        "lattice_MO_bond_A": spec["r_mo_lat"],
        "cus_MOw_bond_A": spec["r_mcus_ow"],
        "enclosing_interface_cutoff_A": spec["r_if_cut"],
        "intramolecular_OH_cutoff_A": spec["r_oh_cut"],
        "delta_chi_MO": delta_chi_mo,
        "pauling_ionicity_MO": ionicity_mo,
        "pauling_ionicity_OH": ionicity_oh,
        "max_stable_flexible_md_dt_fs": 0.5,
        "water_molar_mass_g_mol": 18.01528,
    })

df_features = pd.DataFrame(feature_rows)
df_features.to_csv(FEAT_DIR / "physchem_feature_matrix.csv", index=False)
print("Engineered Physical-Chemical Descriptor Matrix (`physchem_feature_matrix.csv`):")
print(df_features.to_string(index=False))


## Step 2: Engineer Chain-of-Physical-Thought (`<physchem_reasoning>`) SFT Dataset & Counterfactual Physics DPO Pairs
We now construct the three engineered datasets for `PhysChem-finetuning.ipynb`:
1. **`stage1_physchem_world_model.jsonl`**: Teaches the model the explicit physical & chemical laws, bond lengths, cutoff regimes, dimensional conversions, and crystallographic stoichiometries.
2. **`stage2_copt_phys_code_sft.jsonl`**: Pairs prompts with a structured `<physchem_reasoning>` block (deriving chemical species separation, cutoff values in $\text{\AA}$, conservation laws, and ASE array shapes) followed by the verified Python script, plus a list of `physics_critical_substrings` (`"1.25"`, `"3.2"`, `"4.0 * np.pi"`, `"6.022"`, `"18.01528"`, `"1e-24"`, `"mic=True"`, `"neighbor_list"`) that receive **2.0x loss weight** during Stage 2 SFT!
3. **`stage3_counterfactual_physchem_dpo.jsonl`**: Pairs each physically valid response (`chosen`) against **physics-violating counterfactual code** (`rejected`) so Stage 4 DPO actively penalizes physical and chemical violations.


In [ ]:
# ==============================================================================
# CELL 3: ENGINEER COPT SFT RECORDS, PHYSICS TOKEN WEIGHT TAGS & COUNTERFACTUAL DPO PAIRS
# ==============================================================================
stage1_world_model = []
stage2_copt_code_sft = []
stage3_counterfactual_dpo = []

# 1. Stage 1: Chemical Bonding, Crystallography & Statistical Physics World-Model Q&A
for row in feature_rows:
    m = row["metal_symbol"]
    name = row["system_name"]
    s = row["oxide_stoichiometry_s"]
    r_lat = row["lattice_MO_bond_A"]
    r_cus = row["cus_MOw_bond_A"]
    r_if = row["enclosing_interface_cutoff_A"]
    r_oh = row["intramolecular_OH_cutoff_A"]
    ion_mo = row["pauling_ionicity_MO"]

    q_bond = (
        f"What are the exact chemical bond lengths, coordination cutoff distances, and sub-phase partitioning rules "
        f"needed to analyze an interfacial {name} ({m}O_{s}) / liquid H2O simulation in ASE?"
    )
    a_bond = (
        f"For the {name} ({m}O_{s}) / water interface:\n"
        f"1. **Intramolecular Water O-H Covalent Cutoff (`r_OH <= {r_oh} Å`)**: The equilibrium O-H covalent bond length in H2O and surface OH* is ~0.972 Å (with H-O-H bond angle 104.52 deg), whereas intermolecular H-bonds (O...H) start above 1.65 Å. Using `neighbor_list('ij', atoms, cutoff={{('O', 'H'): {r_oh}}})` cleanly isolates water oxygens (`O_w`, bonded to >=1 H) from anhydrous crystal lattice oxygens (`O_slab`, bonded to 0 H within {r_oh} Å).\n"
        f"2. **Crystal Lattice Stoichiometry & Ionicity**: {name} has cation oxidation state +{row['metal_oxidation_state']} and stoichiometric ratio O:{m} = {s}:1 (Pauling bond ionicity f_ionic = {ion_mo:.3f}, bulk {m}-O_slab bond length ~{r_lat} Å). After removing `O_w` and `H`, the remaining `{m}` and `O_slab` atoms form `crystal_indices`.\n"
        f"3. **Coordinatively Unsaturated Site ({m}_cus - O_w) & Enclosing Boundary Shell (`r_cut = {r_if} Å`)**: First-layer chemisorbed water coordinates to surface {m}_cus sites at r({m}_cus - O_w) ~ {r_cus} Å, while surface bridge oxygens (O_br) form H-bonds with water within 2.6-3.1 Å. Therefore, an interfacial cutoff of `{r_if} Å` under 3D periodic boundary conditions (`neighbor_list('ijd', atoms, cutoff={r_if})`) captures the exact first crystal boundary layer enclosing the water phase."
    )
    stage1_world_model.append({"prompt": q_bond, "response": a_bond, "task_type": "physchem_world_model"})

# Add thermodynamic, RDF, MSD, and MD stability laws to Stage 1
law_topics = [
    (
        "Explain the physical laws and mathematical formulas governing (1) planar water mass density rho(z) in g/cm^3, (2) spherical-shell normalized radial distribution function g_AB(r), and (3) continuous residence survival probability P(tau).",
        "1. **Planar Mass Density Profile rho(z) [g/cm^3]**:\n"
        "   - Partition the z-axis into slices of thickness dz (Å). Each slice has volume V_slice = A_xy * dz (Å^3) = A_xy * dz * 10^-24 cm^3.\n"
        "   - Convert water oxygen counts N_Ow(z) to moles via Avogadro's constant N_A = 6.02214076e23 mol^-1 and multiply by molar mass M_H2O = 18.01528 g/mol:\n"
        "     `rho_Ow(z) = ((N_Ow(z) / 6.02214076e23) * 18.01528) / (A_xy * dz * 1e-24)`.\n"
        "   - For hydrogen counts N_Hw(z), divide by `2.0 * N_A` because stoichiometry requires 2 H atoms per H2O molecule.\n"
        "2. **Radial Distribution Function g_AB(r)**:\n"
        "   - Compute pairwise distances under Minimum Image Convention (`d_mat = atoms.get_all_distances(mic=True)`).\n"
        "   - Histogram distances into radial shells [r, r + dr] of volume dV(r) = 4 * pi * r^2 * dr.\n"
        "   - Normalize by the ideal-gas pair expectation `(N_A * N_B / V_cell) * 4 * pi * r^2 * dr`. By Pauli exclusion, g_AB(r) = 0 for r < 0.7 Å.\n"
        "3. **Continuous Survival Probability P(tau)**:\n"
        "   - Tracks the fraction of adsorbate indices S(t_j) present at time t_j that remain continuously adsorbed through all intermediate frames t_j, t_{j+1}, ..., t_j + tau:\n"
        "     `P(tau) = mean_j( |intersection_{k=j}^{j+tau} S(t_k)| / |S(t_j)| )`, which satisfies P(0) = 1.0 and is monotonically non-increasing."
    ),
]
for p_txt, r_txt in law_topics:
    stage1_world_model.append({"prompt": p_txt, "response": r_txt, "task_type": "physchem_world_model"})

# 2. Stage 2 & Stage 3: Chain-of-Physical-Thought (CoPT) Code SFT + Counterfactual Physics DPO
for spec in SYSTEM_PHYSCHEM_SPECS:
    m = spec["metal"]
    name = spec["name"]
    s = spec["stoich"]
    fpath = spec["file"]
    fb = spec["fb"]
    r_oh = spec["r_oh_cut"]
    r_if = spec["r_if_cut"]

    # Task Family A: Enclosing Boundary Layer with CoPT Reasoning Header
    prompt_bound = (
        f"Write a physically rigorous, executable Python script using ASE and NumPy that loads frame 0 of {fpath} "
        f"(with fallback to {fb}), separates water molecules from the {name} crystal slab using chemical bonding rules, "
        f"identifies ONLY the first boundary layer of {name} crystal atoms enclosing the water phase (cutoff {r_if} Å), "
        f"prints a Z-sorted coordinate table, and saves a 2D X-Z side-view projection plot to boundary_enclosing_atoms.png."
    )
    copt_header_bound = (
        f"# [PhysChem Reasoning & Invariants]\n"
        f"# 1. Covalent Species Separation: In {name}/H2O, O belongs to both crystal O_slab and water O_w.\n"
        f"#    Intramolecular covalent r(O-H) ~ 0.97 Å vs H-bond r(O...H) >= 1.65 Å -> use cutoff r_OH = {r_oh} Å.\n"
        f"# 2. Enclosing Crystal Shell: {m}_cus-O_w (~{spec['r_mcus_ow']} Å) and O_br...H_w (~1.8-2.8 Å) lie within r_if = {r_if} Å under 3D PBC.\n"
        f"# 3. ASE API Contract: neighbor_list('ijd', atoms, cutoff={r_if}) returns 3 1D arrays (nl_i, nl_j, nl_d); never index as 2D d[i,j]."
    )
    chosen_bound = f"""```python
{copt_header_bound}
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from ase.io import read
from ase.neighborlist import neighbor_list

file_path = "{fpath}"
fallback_path = "{fb}"
if not os.path.exists(file_path):
    file_path = fallback_path

atoms = read(file_path, index=0)
atoms.set_pbc(True)
positions = atoms.get_positions()
symbols = np.array(atoms.get_chemical_symbols())

# Step 1: Separate covalently bonded water (O_w, H) from {name} crystal slab ({m}, O_slab)
i_oh, j_oh = neighbor_list('ij', atoms, cutoff={{('O', 'H'): {r_oh}}}, self_interaction=False)
o_w_set = set(np.unique(i_oh))
h_set = set(np.where(symbols == 'H')[0])
water_indices = sorted(h_set | o_w_set)
water_set = set(water_indices)

crystal_indices = sorted([
    idx for idx, sym in enumerate(symbols)
    if sym in ('{m}', 'O') and idx not in water_set
])
crystal_set = set(crystal_indices)

# Step 2: Identify first boundary layer of crystal atoms enclosing water under PBC (cutoff = {r_if} Å)
nl_i, nl_j, nl_d = neighbor_list('ijd', atoms, cutoff={r_if}, self_interaction=False)
min_dist_by_crystal = {{}}
for i_atom, j_atom, dist in zip(nl_i, nl_j, nl_d):
    if i_atom in crystal_set and j_atom in water_set:
        if i_atom not in min_dist_by_crystal or dist < min_dist_by_crystal[i_atom]:
            min_dist_by_crystal[int(i_atom)] = float(dist)

first_boundary_indices = sorted(min_dist_by_crystal.keys(), key=lambda idx: positions[idx, 2])

print(f"Total Atoms: {{len(atoms)}}")
print(f"Crystal Slab Atoms ({m} + O_slab): {{len(crystal_indices)}}")
print(f"Adsorbed Water Atoms (O_w + H): {{len(water_indices)}}")
print(f"First Boundary Crystal Atoms Enclosing Water: {{len(first_boundary_indices)}}\\n")

header = f"{{'Atom_ID':<8}} | {{'Symbol':<6}} | {{'X (Å)':>10}} | {{'Y (Å)':>10}} | {{'Z (Å)':>10}} | {{'Min_Dist_to_Water (Å)':>21}}"
print(header)
print("-" * len(header))
for idx in first_boundary_indices:
    x, y, z = positions[idx]
    print(f"{{idx:<8d}} | {{symbols[idx]:<6s}} | {{x:>10.4f}} | {{y:>10.4f}} | {{z:>10.4f}} | {{min_dist_by_crystal[idx]:>21.4f}}")

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(positions[crystal_indices, 0], positions[crystal_indices, 2], s=18, color='slategray', alpha=0.5, label='{name} Crystal Bulk')
ax.scatter(positions[water_indices, 0], positions[water_indices, 2], s=22, color='dodgerblue', alpha=0.6, label='Water Phase (O_w, H)')
if first_boundary_indices:
    ax.scatter(positions[first_boundary_indices, 0], positions[first_boundary_indices, 2], s=75, color='crimson', edgecolors='black', linewidths=1.2, zorder=5, label='First Enclosing Boundary Layer')
ax.set_xlabel('X Position (Å)')
ax.set_ylabel('Z Position (Å)')
ax.set_title('2D X-Z Side-View Projection: {name} Boundary Enclosing Water')
ax.legend(loc='best')
ax.grid(True, linestyle='--', alpha=0.5)
plt.savefig('boundary_enclosing_atoms.png', dpi=300, bbox_inches='tight')
plt.close()
```"""

    # Counterfactual Physics-Violating Negative 1: Wrong O-H cutoff (2.1 Å merges H-bonds), no PBC MIC, Euclidean norm
    rejected_phys_bound = f"""```python
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read

atoms = read("{fpath}", index=0)
positions = atoms.get_positions()
symbols = np.array(atoms.get_chemical_symbols())

# PHYSICS VIOLATION 1: Treats ALL O atoms as water or uses unphysical 2.1 Å covalent cutoff without PBC!
water_indices = [i for i, s in enumerate(symbols) if s in ('O', 'H')]
crystal_indices = [i for i, s in enumerate(symbols) if s == '{m}']

# PHYSICS VIOLATION 2: Naive Euclidean distance np.linalg.norm without Minimum Image Convention (PBC)
first_boundary_indices = []
for i in crystal_indices:
    for j in water_indices:
        dist = np.linalg.norm(positions[i] - positions[j])
        if dist <= {r_if}:
            first_boundary_indices.append(i)
            break

# PHYSICS VIOLATION 3: Plots X-Y top view instead of X-Z side view
plt.scatter(positions[:, 0], positions[:, 1])
plt.savefig('boundary_enclosing_atoms.png')
```"""

    stage2_copt_code_sft.append({
        "prompt": prompt_bound,
        "response": chosen_bound,
        "task_type": "copt_boundary_code",
        "physics_critical_tokens": [str(r_oh), str(r_if), "neighbor_list", "set_pbc(True)", "water_set", "crystal_set"],
    })
    stage3_counterfactual_dpo.append({
        "prompt": prompt_bound,
        "chosen": chosen_bound,
        "rejected": rejected_phys_bound,
        "violation_type": "cutoff_confluence_and_missing_pbc_mic",
    })

    # Task Family B: Planar Water Mass Density Profile rho(z) + Spherical-Shell Normalized RDF g(r)
    prompt_rdf = (
        f"Write a physically rigorous Python script using ASE, NumPy, and Matplotlib (no pwtools) to analyze {fpath} "
        f"(fallback: {fb}) for {name}. Compute (1) the planar water mass density profile rho(z) in g/cm^3 using exact "
        f"Avogadro/molar-mass dimensional conversion and (2) the spherical-shell normalized radial distribution functions "
        f"g({m}_sur - O_w) and g(O_slab - O_w) up to 6.0 Å along with running coordination numbers CN(r). Save water_density_and_rdf.png."
    )
    chosen_rdf = f"""```python
# [PhysChem Reasoning & Invariants]
# 1. Dimensional Conversion for rho(z) [g/cm^3]:
#    V_slice [cm^3] = A_xy [Å^2] * dz [Å] * 1e-24 [cm^3/Å^3].
#    rho_Ow [g/cm^3] = ((N_Ow / N_A) * 18.01528) / V_slice; rho_Hw uses (N_Hw / (2 * N_A)) by 2:1 H:O stoichiometry.
# 2. Spherical Shell RDF Normalization:
#    dV(r) = 4 * pi * r^2 * dr. Ideal-gas pair count = (N_src * N_dst / V_cell) * dV(r).
#    Pauli repulsion guarantees g(r) = 0 for r < 0.7 Å.
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from ase.io import read
from ase.neighborlist import neighbor_list

file_path = "{fpath}"
fallback_path = "{fb}"
if not os.path.exists(file_path):
    file_path = fallback_path

raw = read(file_path, index=':')
frames = raw if isinstance(raw, list) else [raw]

atoms0 = frames[0].copy()
atoms0.wrap()
atoms0.set_pbc(True)
pos0 = atoms0.get_positions()
sym0 = np.array(atoms0.get_chemical_symbols())

i_oh, _ = neighbor_list('ij', atoms0, cutoff={{('O', 'H'): {r_oh}}}, self_interaction=False)
o_w_set = set(np.unique(i_oh))
o_w_indices = sorted(o_w_set)
h_w_indices = sorted(np.where(sym0 == 'H')[0].tolist())
m_indices = sorted(np.where(sym0 == '{m}')[0].tolist())
o_slab_indices = sorted([i for i, s in enumerate(sym0) if s == 'O' and i not in o_w_set])

m_z_max = float(pos0[m_indices, 2].max())
m_sur_indices = [i for i in m_indices if pos0[i, 2] >= m_z_max - 1.5]
o_sur_indices = [i for i in o_slab_indices if pos0[i, 2] >= m_z_max - 1.5]

cell = atoms0.get_cell()
cell_vol = float(atoms0.get_volume())
area_xy = float(np.linalg.norm(np.cross(cell[0], cell[1])))
dz = 0.10
z_bins = np.arange(0.0, float(cell[2, 2]) + dz, dz)
z_centers = 0.5 * (z_bins[:-1] + z_bins[1:])
slice_vol_cm3 = area_xy * dz * 1e-24
avogadro = 6.02214076e23
molar_mass_h2o = 18.01528

dr = 0.05
rmax = 6.0
r_bins = np.arange(0.0, rmax + dr, dr)
r_centers = 0.5 * (r_bins[:-1] + r_bins[1:])
shell_volumes = 4.0 * np.pi * (r_centers ** 2) * dr

rho_ow_list, rho_hw_list = [], []
g_mow_list, g_osow_list = [], []

for frame in frames:
    frame.wrap()
    frame.set_pbc(True)
    f_pos = frame.get_positions()
    c_ow, _ = np.histogram(f_pos[o_w_indices, 2], bins=z_bins)
    c_hw, _ = np.histogram(f_pos[h_w_indices, 2], bins=z_bins)
    rho_ow_list.append(((c_ow / avogadro) * molar_mass_h2o) / slice_vol_cm3)
    rho_hw_list.append(((c_hw / (2.0 * avogadro)) * molar_mass_h2o) / slice_vol_cm3)

    d_mat = frame.get_all_distances(mic=True)
    for src, dst, store in [(m_sur_indices, o_w_indices, g_mow_list), (o_sur_indices, o_w_indices, g_osow_list)]:
        if src and dst:
            sub_d = d_mat[np.ix_(src, dst)].ravel()
            sub_d = sub_d[sub_d > 1e-5]
            hist, _ = np.histogram(sub_d, bins=r_bins)
            norm = (len(src) * len(dst) / cell_vol) * shell_volumes
            store.append(hist / np.maximum(norm, 1e-12))
        else:
            store.append(np.zeros_like(r_centers))

rho_ow = np.mean(rho_ow_list, axis=0)
rho_hw = np.mean(rho_hw_list, axis=0)
g_mow = np.mean(g_mow_list, axis=0)
g_osow = np.mean(g_osow_list, axis=0)

trapz_fn = getattr(np, 'trapezoid', getattr(np, 'trapz', None))
rho_num_ow = len(o_w_indices) / cell_vol
cn_mow = np.array([
    4.0 * np.pi * rho_num_ow * trapz_fn(g_mow[:k+1] * (r_centers[:k+1] ** 2), r_centers[:k+1])
    for k in range(len(r_centers))
])

print(f"Peak Water Density rho_Ow: {{rho_ow.max():.3f}} g/cm^3 | Peak g({m}_sur-O_w): {{g_mow.max():.3f}}")
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
axs[0].plot(z_centers, rho_ow, label='rho_Ow (g/cm^3)', color='navy')
axs[0].plot(z_centers, rho_hw, label='rho_Hw (g/cm^3)', color='dodgerblue', ls='--')
axs[0].set_xlabel('Z Coordinate (Å)')
axs[0].set_ylabel('Mass Density (g/cm^3)')
axs[0].legend()
axs[1].plot(r_centers, g_mow, label='g({m}_sur - O_w)', color='crimson')
axs[1].plot(r_centers, g_osow, label='g(O_slab - O_w)', color='darkgreen')
axs[1].set_xlabel('Distance r (Å)')
axs[1].set_ylabel('g(r)')
axs[1].legend()
plt.tight_layout()
plt.savefig('water_density_and_rdf.png', dpi=300, bbox_inches='tight')
plt.close()
```"""

    # Counterfactual Physics-Violating Negative 2: Raw unnormalized histogram as RDF & missing 1e-24 / Avogadro in density
    rejected_phys_rdf = f"""```python
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read

atoms = read("{fpath}", index=0)
pos = atoms.get_positions()
# PHYSICS VIOLATION 1: Raw histogram without Avogadro (6.022e23), molar mass (18.015), or 1e-24 cm^3/Å^3 conversion!
rho_z, bins = np.histogram(pos[:, 2], bins=50)
# PHYSICS VIOLATION 2: Divides RDF by constant instead of spherical shell volume 4 * pi * r^2 * dr!
d_mat = atoms.get_all_distances(mic=False)
g_r, r_bins = np.histogram(d_mat.ravel(), bins=50, range=(0, 6.0))
plt.plot(g_r)
plt.savefig('water_density_and_rdf.png')
```"""

    stage2_copt_code_sft.append({
        "prompt": prompt_rdf,
        "response": chosen_rdf,
        "task_type": "copt_density_rdf_code",
        "physics_critical_tokens": ["6.02214076e23", "18.01528", "1e-24", "4.0 * np.pi * (r_centers ** 2) * dr", "mic=True"],
    })
    stage3_counterfactual_dpo.append({
        "prompt": prompt_rdf,
        "chosen": chosen_rdf,
        "rejected": rejected_phys_rdf,
        "violation_type": "unnormalized_spherical_shell_and_wrong_density_units",
    })

# Verify AST validity of all engineered Stage 2 code blocks
for rec in stage2_copt_code_sft:
    code_only = re.search(r"```python\s*(.*?)```", rec["response"], flags=re.DOTALL).group(1)
    ast.parse(code_only)

# Save engineered JSONL files for PhysChem-finetuning.ipynb
def write_jsonl(path: Path, data: list):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

write_jsonl(FEAT_DIR / "stage1_physchem_world_model.jsonl", stage1_world_model * 20)
write_jsonl(FEAT_DIR / "stage2_copt_phys_code_sft.jsonl", stage2_copt_code_sft * 25)
write_jsonl(FEAT_DIR / "stage3_counterfactual_physchem_dpo.jsonl", stage3_counterfactual_dpo * 20)

print(f"\nFeature Engineering Complete! Saved to {FEAT_DIR}:")
print(f"  - stage1_physchem_world_model.jsonl:      {len(stage1_world_model) * 20} records")
print(f"  - stage2_copt_phys_code_sft.jsonl:        {len(stage2_copt_code_sft) * 25} records")
print(f"  - stage3_counterfactual_physchem_dpo.jsonl: {len(stage3_counterfactual_dpo) * 20} records")
print(f"  - physchem_feature_matrix.csv:            {len(df_features)} rows")
